# Name: Prathamesh Arvind Jadhav   
#Roll No: 4059  

In [5]:
!pip install transformers datasets torch scikit-learn pandas numpy tqdm

In [7]:
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
from tqdm import tqdm


In [8]:
df = pd.read_csv("/content/drive/MyDrive/fake_news_dataset.csv")



In [9]:
df = df.dropna(subset=['text', 'label'])
texts = df['text'].tolist()
labels = df['label'].tolist()


In [10]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)


In [12]:
MODEL_NAME = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def encode_data(texts, labels):
    encodings = tokenizer(
        texts, truncation=True, padding=True, max_length=256, return_tensors='pt'
    )
    # Convert labels to numerical format
    numerical_labels = [1 if label == 'Real' else 0 for label in labels]
    return encodings, torch.tensor(numerical_labels)

train_encodings, train_labels = encode_data(train_texts, train_labels)
val_encodings, val_labels = encode_data(val_texts, val_labels)

In [13]:
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FakeNewsDataset(train_encodings, train_labels)
val_dataset = FakeNewsDataset(val_encodings, val_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)



In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [15]:
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 2

total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps
)


In [16]:
for epoch in range(epochs):
    print(f"\n===== Epoch {epoch + 1} / {epochs} =====")
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc="Training"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Training loss: {avg_loss:.4f}")




===== Epoch 1 / 2 =====


Training: 100%|██████████| 400/400 [19:53<00:00,  2.98s/it]


Training loss: 0.7080

===== Epoch 2 / 2 =====


Training: 100%|██████████| 400/400 [19:14<00:00,  2.89s/it]

Training loss: 0.7004


In [18]:
model.eval()
preds, true_labels = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="Validating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

acc = accuracy_score(true_labels, preds)
print(f"Validation Accuracy: {acc:.4f}")
print(classification_report(true_labels, preds, target_names=["Genuine", "Fake"]))

Validating: 100%|██████████| 100/100 [01:40<00:00,  1.01s/it]

Validation Accuracy: 0.4863
              precision    recall  f1-score   support

     Genuine       0.00      0.00      0.00       411
        Fake       0.49      1.00      0.65       389

    accuracy                           0.49       800
   macro avg       0.24      0.50      0.33       800
weighted avg       0.24      0.49      0.32       800




/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [19]:
model.save_pretrained("bert_fake_news_model")
tokenizer.save_pretrained("bert_fake_news_model")
print(" Model and tokenizer saved successfully!")


 Model and tokenizer saved successfully!


In [20]:
from transformers import BertTokenizer, BertForSequenceClassification


model_path = "bert_fake_news_model"
model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)
model.to(device)
model.eval()


sample_texts = [
    "Breaking news: Scientists have discovered a new planet that supports life.",
    "Government confirms aliens have landed in New York City!",
    "The Prime Minister inaugurated a new AI research center today in Mumbai."
]


inputs = tokenizer(sample_texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)


with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1).cpu().numpy()

for text, pred in zip(sample_texts, predictions):
    label = "Fake" if pred == 1 else "Genuine"
    print(f"\n📰 Text: {text}\n→ Prediction: {label}")



📰 Text: Breaking news: Scientists have discovered a new planet that supports life.
→ Prediction: Genuine

📰 Text: Government confirms aliens have landed in New York City!
→ Prediction: Genuine

📰 Text: The Prime Minister inaugurated a new AI research center today in Mumbai.
→ Prediction: Genuine
